## Misión Asíncrona: Cazando al Falso Influencer

## Fase 1: Data Engineering (La Explosión)

In [28]:
import pandas as pd
import networkx as nx

df = pd.read_csv('export_api_influencers.csv')
print("Crudo: ", df.shape)
df.head(10)

Crudo:  (75, 4)


,Usuario,Ubicacion,Fecha_Creacion,Cuentas_Que_Sigue
0,Bot_5,India,2023-05-06,AMATEUR_1
1,fan_6,Mexico,2023-07-14,"GARNACHO, MESSI"
2,Bot_11,India,2023-02-14,AMATEUR_1
3,Bot_1,India,2023-11-24,AMATEUR_1
4,Bot_36,India,2023-06-24,AMATEUR_1
5,Fan_4,Mexico,2023-05-29,GARNACHO
6,Bot_29,India,2023-08-18,AMATEUR_1
7,Bot_13,India,2023-08-05,AMATEUR_1
8,Bot_18,India,2023-04-30,AMATEUR_1
9,Fan_7,Mexico,2023-11-22,GARNACHO


In [29]:
df = df.dropna(subset=['Cuentas_Que_Sigue'])
print("Nulos eliminados: ", df.shape)
df['Usuario'] = df['Usuario'].str.upper().str.strip()

df = df.drop_duplicates(subset='Usuario', keep='first')
print("Duplicados eliminados: ", df.shape)

Nulos eliminados:  (71, 4)
Duplicados eliminados:  (65, 4)


In [30]:
df['Cuentas_Que_Sigue'] = df['Cuentas_Que_Sigue'].str.split(',')

edges = (df[['Usuario', 'Cuentas_Que_Sigue']].explode('Cuentas_Que_Sigue').rename(columns={'Usuario': 'Origen', 'Cuentas_Que_Sigue': 'Destino'}))

print("Explode: ", edges.shape)
edges.head(10)

Explode:  (98, 2)


,Origen,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
1,FAN_6,MESSI
2,BOT_11,AMATEUR_1
3,BOT_1,AMATEUR_1
4,BOT_36,AMATEUR_1
5,FAN_4,GARNACHO
6,BOT_29,AMATEUR_1
7,BOT_13,AMATEUR_1
8,BOT_18,AMATEUR_1


In [31]:
edges['Destino'] = edges['Destino'].str.upper().str.strip()

# Self loops
print("Self loops: ", (edges['Origen'] == edges['Destino']).sum())
edges = edges[edges['Origen'] != edges['Destino']].reset_index(drop=True)

print("Artistas finales: ", edges.shape)
edges.head(10)

Self loops:  1
Artistas finales:  (97, 2)


,Origen,Destino
0,BOT_5,AMATEUR_1
1,FAN_6,GARNACHO
2,FAN_6,MESSI
3,BOT_11,AMATEUR_1
4,BOT_1,AMATEUR_1
5,BOT_36,AMATEUR_1
6,FAN_4,GARNACHO
7,BOT_29,AMATEUR_1
8,BOT_13,AMATEUR_1
9,BOT_18,AMATEUR_1


## Fase 2: El Engaño de la Popularidad

In [32]:
G = nx.from_pandas_edgelist(edges, source='Origen', target='Destino', create_using=nx.DiGraph)
print("Nodos: ", G.number_of_nodes(), " Artistas: ", G.number_of_edges())

in_cent = nx.in_degree_centrality(G)
top3 = sorted(in_cent.items(), key=lambda x: x[1], reverse=True)[:3]

print("Top 3: ")
for user, score in top3: print(f'  {user:<12} centralidad = {score:.4f}  (seguidores = {G.in_degree(user)})')

Nodos:  66  Artistas:  97
Top 3: 
  AMATEUR_1    centralidad = 0.7231  (seguidores = 47)
  GARNACHO     centralidad = 0.2000  (seguidores = 13)
  MESSI        centralidad = 0.1538  (seguidores = 10)


Pregunta: A simple vista, ¿quién parece ser el rey de la red?

AMATEUR_1 ya que tiene casi el cuádruple de seguidores que GARNACHO

## Fase 3: La Verdad Topológica (Las Métricas)

In [35]:
pr = nx.pagerank(G)

top = sorted(pr.items(), key=lambda x: x[1], reverse=True)[:8]
print("Top 8: ")
for user, score in top: print(f'{user:<12} {score:.6f}')

print(f"\nAMATEUR_1: {pr['AMATEUR_1']:.6f}")
print(f"GARNACHO : {pr['GARNACHO']:.6f}")

Top 8: 
AMATEUR_1    0.196941
DE_PAUL      0.088285
GARNACHO     0.084288
SUAREZ       0.079382
OTAMENDI     0.068620
DI_MARIA     0.065470
MESSI        0.058973
NEYMAR       0.052540

AMATEUR_1: 0.196941
GARNACHO : 0.084288


In [37]:
U = G.to_undirected()
clust = nx.clustering(U)

print(f"Clustering AMATEUR_1: {clust['AMATEUR_1']}")
print(f"Clustering GARNACHO : {clust['GARNACHO']}")
print(f"\nGrado (no dirigido) -> AMATEUR_1: {U.degree('AMATEUR_1')}  GARNACHO: {U.degree('GARNACHO')}")

Clustering AMATEUR_1: 0
Clustering GARNACHO : 0.13333333333333333

Grado (no dirigido) -> AMATEUR_1: 47  GARNACHO: 15


#### ¿Quién compró bots y por qué?

AMATEUR_1, ya que su clustering es exactamente 0, lo que nos dice que ninguno de sus seguidores esta conectado con otro.


#### ¿Quién es el influencer real y por qué?

GARNACHO, ya que su clustering es 0.1333, lo que nos dice que su vecindario sí forma triángulos, es decir, parte de sus seguidores también están conectados entre sí.